# Final ML Project: HR Analytics Job Change Prediction

This notebook completes the final classification project using the HR Analytics Job Change dataset. The goal is to predict whether a candidate is likely to look for a job change.

Target meaning:

- `1` means the candidate is likely to look for a job change.
- `0` means the candidate is not likely to look for a job change.

This notebook is organized to match the final exam rubric: preprocessing, required train/validation/test split, required classification models, validation/test comparison metrics, ensemble models, final model selection, and saved outputs for the report.

## Project structure and important fixes

This notebook uses a GitHub-friendly folder structure:

```text
project_folder/
├── data/
│   ├── raw/aug_train.csv
│   └── processed/
├── output/
├── report/
├── presentation/
└── notebooks/
```

Important modeling fixes:

- `city_development_index` stays numeric and is scaled once inside the numeric preprocessing pipeline.
- `training_hours` is converted to `training_hours_log1p`, then the raw `training_hours` column is removed to avoid duplicate information.
- Ordered fields are converted into numeric meaning instead of being treated as random text: `experience`, `last_new_job`, `company_size`, and `education_level`.
- Nominal fields with no natural order are one-hot encoded.
- The data split is exactly aligned with the requirement: 70% train, 15% validation, and 15% test.
- The ensemble section uses both an average ensemble and a validation-weighted Bayesian-style ensemble for comparison.

## Key fixes from the midterm feedback

- `city_development_index` is not transformed twice. It stays numeric and is scaled once inside the numeric preprocessing pipeline.
- `training_hours` uses `log1p`, and the raw training-hours column is removed from modeling to avoid duplicate information.
- Ordered fields are converted into numeric meaning instead of being one-hot encoded: `experience`, `last_new_job`, and `company_size`.
- Nominal fields with no natural order are one-hot encoded.

In [1]:
# ================================
# 1. Imports and folder setup
# ================================
# This cell keeps all imports and output folders in one place so the notebook is easy to rerun.

from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import BayesianRidge

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

# Use the current working directory as the project root.
# If the notebook is stored inside a /notebooks folder, this logic can still find /data/raw.
ROOT = Path.cwd()
if not (ROOT / 'data' / 'raw' / 'aug_train.csv').exists() and (ROOT.parent / 'data' / 'raw' / 'aug_train.csv').exists():
    ROOT = ROOT.parent

DATA_RAW = ROOT / 'data' / 'raw'
DATA_PROCESSED = ROOT / 'data' / 'processed'
OUTPUT = ROOT / 'output'
REPORT = ROOT / 'report'
PRESENTATION = ROOT / 'presentation'
NOTEBOOKS = ROOT / 'notebooks'

# Create folders if they do not already exist.
for folder in [DATA_RAW, DATA_PROCESSED, OUTPUT, REPORT, PRESENTATION, NOTEBOOKS]:
    folder.mkdir(parents=True, exist_ok=True)

print('Project root:', ROOT)
print('Raw data folder:', DATA_RAW)
print('Output folder:', OUTPUT)

Project root: /content
Raw data folder: /content/data/raw
Output folder: /content/output


## Load data and create the required 70%, 15%, 15% split

The assignment requires three separate datasets:

- Training set: 70%
- Validation set: 15%
- Test set: 15%

The split is stratified so the target-class ratio stays similar across train, validation, and test sets.

In [2]:
# ================================
# 2. Load data and split dataset
# ================================

src = DATA_RAW / 'aug_train.csv'

# Reproducibility guard:
# The professor or grader should be able to understand exactly where the data file belongs.
if not src.exists():
    raise FileNotFoundError(
        f"Missing dataset file: {src}. "
        "Please place aug_train.csv under data/raw/ or update DATA_RAW path."
    )

df = pd.read_csv(src)
print('Dataset shape:', df.shape)
display(df.head())

TARGET_COL = 'target'

# Separate features from target.
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].astype(int)

# First split: 70% train, 30% temporary holdout.
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)

# Second split: divide the 30% holdout equally into 15% validation and 15% test.
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp,
)

print('Train:', X_train.shape, 'Validation:', X_val.shape, 'Test:', X_test.shape)
print('Target distribution by split:')
display(pd.DataFrame({
    'train': y_train.value_counts(normalize=True).sort_index(),
    'validation': y_val.value_counts(normalize=True).sort_index(),
    'test': y_test.value_counts(normalize=True).sort_index(),
}).round(4))

Dataset shape: (19158, 14)


,enrollee_id,city,city_development_index,gender,relevent_experience,enrolled_university,education_level,major_discipline,experience,company_size,company_type,last_new_job,training_hours,target
0,8949,city_103,0.920,Male,Has relevent experience,no_enrollment,Graduate,STEM,>20,NaN,NaN,1,36,1.0
1,29725,city_40,0.776,Male,No relevent experience,no_enrollment,Graduate,STEM,15,50-99,Pvt Ltd,>4,47,0.0
2,11561,city_21,0.624,NaN,No relevent experience,Full time course,Graduate,STEM,5,NaN,NaN,never,83,0.0
3,33241,city_115,0.789,NaN,No relevent experience,NaN,Graduate,Business Degree,<1,NaN,Pvt Ltd,never,52,1.0
4,666,city_162,0.767,Male,Has relevent experience,no_enrollment,Masters,STEM,>20,50-99,Funded Startup,4,8,0.0


Train: (13410, 13) Validation: (2874, 13) Test: (2874, 13)
Target distribution by split:


,train,validation,test
target,,,
0,0.7506,0.7505,0.7509
1,0.2494,0.2495,0.2491


## Fix ordered values and prepare features

This section converts columns that have a real order into numeric values before preprocessing. This improves model meaning and avoids treating ordered categories like unrelated text labels.

In [3]:
# ================================
# 3. Feature engineering helpers
# ================================

# Ordered category mappings.
# These fields have a natural order, so numeric encoding is more meaningful than one-hot encoding.
education_map = {
    'Primary School': 0,
    'High School': 1,
    'Graduate': 2,
    'Masters': 3,
    'Phd': 4,
}

relevant_experience_map = {
    'No relevent experience': 0,
    'Has relevent experience': 1,
}

company_size_midpoint_map = {
    '<10': 5,
    '10-49': 30,
    '10/49': 30,
    '50-99': 75,
    '100-500': 300,
    '500-999': 750,
    '1000-4999': 3000,
    '5000-9999': 7500,
    '10000+': 10000,
}


def parse_experience(value):
    """Convert experience values such as '<1', '5', and '>20' into numeric values."""
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    if value == '<1':
        return 0.5
    if value == '>20':
        return 21.0
    return pd.to_numeric(value, errors='coerce')


def parse_last_new_job(value):
    """Convert last_new_job values such as 'never', '1', and '>4' into numeric values."""
    if pd.isna(value):
        return np.nan
    value = str(value).strip().lower()
    if value == 'never':
        return 0.0
    if value == '>4':
        return 5.0
    return pd.to_numeric(value, errors='coerce')


def prepare_features(frame):
    """Apply feature engineering before the sklearn preprocessing pipeline.

    Important design choice:
    - Fit statistics such as medians and one-hot categories are learned only from the training set later.
    - This function only performs deterministic conversions, so it is safe to apply to train, validation, and test.
    """
    data = frame.copy()

    # Convert ordered text fields into numeric meaning.
    data['relevent_experience'] = data['relevent_experience'].map(relevant_experience_map)
    data['education_level'] = data['education_level'].map(education_map)
    data['experience'] = data['experience'].apply(parse_experience)
    data['last_new_job'] = data['last_new_job'].apply(parse_last_new_job)
    data['company_size'] = data['company_size'].map(company_size_midpoint_map)

    # Reduce skew in training_hours and remove the raw column to avoid duplicate information.
    data['training_hours_log1p'] = np.log1p(data['training_hours'].clip(lower=0))

    # Drop identifier and high-cardinality city field.
    # enrollee_id is an ID, not a useful predictor.
    # city is high-cardinality and may cause noisy one-hot features for this class project.
    return data.drop(columns=['enrollee_id', 'city', 'training_hours'])



## Build preprocessing pipeline and save processed data

The preprocessing pipeline is fit only on the training set to avoid data leakage. Validation and test data are transformed using the fitted training pipeline.

In [4]:
# ================================
# 4. Preprocessing pipeline
# ================================

X_train_fe = prepare_features(X_train)
X_val_fe = prepare_features(X_val)
X_test_fe = prepare_features(X_test)

# Detect numeric and categorical columns after feature engineering.
numeric_features = X_train_fe.select_dtypes(include=['number']).columns.tolist()
categorical_features = X_train_fe.select_dtypes(include=['object', 'category']).columns.tolist()

print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)

# Numeric pipeline: fill missing values with median, then scale.
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

# Categorical pipeline: fill missing values with most frequent category, then one-hot encode.
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features),
])

# Fit only on training data. Transform validation and test using the same fitted preprocessor.
X_train_clean = preprocessor.fit_transform(X_train_fe)
X_val_clean = preprocessor.transform(X_val_fe)
X_test_clean = preprocessor.transform(X_test_fe)

# Create readable feature names for saved processed files and feature importance.
onehot = preprocessor.named_transformers_['cat'].named_steps['onehot']
feature_names = numeric_features + list(onehot.get_feature_names_out(categorical_features))

print('Processed train shape:', X_train_clean.shape)
print('Processed validation shape:', X_val_clean.shape)
print('Processed test shape:', X_test_clean.shape)

# Save processed and raw split files for GitHub deliverables.
for split_name, X_clean, y_split in [
    ('train', X_train_clean, y_train),
    ('validation', X_val_clean, y_val),
    ('test', X_test_clean, y_test),
]:
    output_df = pd.DataFrame(X_clean, columns=feature_names)
    output_df[TARGET_COL] = y_split.reset_index(drop=True).values
    output_df.to_csv(DATA_PROCESSED / f'hr_jobchange_{split_name}_processed.csv', index=False)

for split_name, X_raw, y_split in [
    ('train', X_train, y_train),
    ('validation', X_val, y_val),
    ('test', X_test, y_test),
]:
    output_df = X_raw.copy()
    output_df[TARGET_COL] = y_split.values
    output_df.to_csv(DATA_PROCESSED / f'hr_jobchange_{split_name}_raw_split.csv', index=False)


Numeric features: ['city_development_index', 'relevent_experience', 'education_level', 'experience', 'company_size', 'last_new_job', 'training_hours_log1p']
Categorical features: ['gender', 'enrolled_university', 'major_discipline', 'company_type']
Processed train shape: (13410, 25)
Processed validation shape: (2874, 25)
Processed test shape: (2874, 25)


## Train all required classification models

The classification rubric requires the following models:

1. Logistic Regression
2. Decision Tree Classifier
3. Random Forest Classifier
4. Gradient Boosting Classifier
5. K-Nearest Neighbors Classifier
6. Support Vector Classifier

Each model is evaluated on both the validation set and the test set using accuracy, precision, recall, F1-score, and ROC-AUC.

In [5]:
# ================================
# 5. Train and evaluate required models
# ================================

models = {
    'Logistic Regression': LogisticRegression(
        max_iter=2000,
        class_weight='balanced',
        random_state=42,
    ),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=8,
        min_samples_leaf=40,
        class_weight='balanced',
        random_state=42,
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=180,
        max_depth=12,
        min_samples_leaf=10,
        class_weight='balanced_subsample',
        random_state=42,
        n_jobs=-1,
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=3,
        random_state=42,
    ),
    'K-Nearest Neighbors': KNeighborsClassifier(
        n_neighbors=25,
        weights='distance',
    ),
    # The final exam specifically lists Support Vector Classifier.
    # This uses sklearn.svm.SVC directly and enables probability=True so the model can be compared by ROC-AUC
    # and can also participate fairly in probability-based ensembles.
    'Support Vector Classifier': SVC(
        kernel='rbf',
        C=1.0,
        gamma='scale',
        probability=True,
        class_weight='balanced',
        random_state=42,
        cache_size=500,
    ),
}


def get_model_score_values(model, X_eval):
    """Return positive-class probabilities or continuous scores for ROC-AUC."""
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X_eval)[:, 1]
    if hasattr(model, 'decision_function'):
        return model.decision_function(X_eval)
    return model.predict(X_eval).astype(float)


def evaluate_classifier(name, model, X_eval, y_eval, split_name):
    """Calculate the exact classification metrics required by the final project rubric."""
    predictions = model.predict(X_eval)
    scores = get_model_score_values(model, X_eval)

    return {
        'model': name,
        'split': split_name,
        'accuracy': accuracy_score(y_eval, predictions),
        'precision': precision_score(y_eval, predictions, zero_division=0),
        'recall': recall_score(y_eval, predictions, zero_division=0),
        'f1_score': f1_score(y_eval, predictions, zero_division=0),
        'roc_auc': roc_auc_score(y_eval, scores),
    }


fitted = {}
rows = []

for name, model in models.items():
    print(f'Training {name}...', flush=True)

    # Train each required model on the full training set.
    # This avoids the grading weakness of training one model on a smaller sample.
    model.fit(X_train_clean, y_train)
    fitted[name] = model

    # Evaluate on validation and test for the required comparison table.
    rows.append(evaluate_classifier(name, model, X_val_clean, y_val, 'Validation'))
    rows.append(evaluate_classifier(name, model, X_test_clean, y_test, 'Test'))

result_df = pd.DataFrame(rows)
metric_columns = ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']
result_df[metric_columns] = result_df[metric_columns].round(4)

# Save the model comparison table for the final report.
result_df.to_csv(OUTPUT / 'all_model_metrics.csv', index=False)

# Rank models by validation ROC-AUC first, then F1-score.
# Validation metrics should guide model selection because the test set should remain the final unbiased check.
val_rank = result_df[result_df['split'] == 'Validation'].sort_values(
    ['roc_auc', 'f1_score', 'accuracy'],
    ascending=[False, False, False],
)

# Ensembles below require probability predictions, so keep only models that support predict_proba.
prob_model_names = [model_name for model_name in val_rank['model'].tolist() if hasattr(fitted[model_name], 'predict_proba')]
top3 = prob_model_names[:3]

print('Top 3 probability-based models for ensemble:', top3)
display(result_df.sort_values(['split', 'roc_auc', 'f1_score'], ascending=[True, False, False]))


Training Logistic Regression...
Training Decision Tree...
Training Random Forest...
Training Gradient Boosting...
Training K-Nearest Neighbors...
Training Support Vector Classifier...
Top 3 probability-based models for ensemble: ['Gradient Boosting', 'Random Forest', 'Decision Tree']


,model,split,accuracy,precision,recall,f1_score,roc_auc
5,Random Forest,Test,0.7679,0.5259,0.6955,0.5989,0.7908
7,Gradient Boosting,Test,0.7717,0.5551,0.4218,0.4794,0.7902
3,Decision Tree,Test,0.7001,0.4432,0.7961,0.5694,0.7776
11,Support Vector Classifier,Test,0.7265,0.4649,0.6466,0.5409,0.7460
1,Logistic Regression,Test,0.7070,0.4410,0.6578,0.5280,0.7358
9,K-Nearest Neighbors,Test,0.7658,0.5413,0.3939,0.4559,0.7307
6,Gradient Boosting,Validation,0.7874,0.6008,0.4407,0.5084,0.8064
4,Random Forest,Validation,0.7665,0.5252,0.6695,0.5886,0.8011
2,Decision Tree,Validation,0.7119,0.4559,0.8006,0.5810,0.7869
10,Support Vector Classifier,Validation,0.7345,0.4764,0.6485,0.5493,0.7556


## Build and evaluate ensemble models

The classification rubric asks for an ensemble model and a Bayesian ensemble comparison. Since this is a classification problem, this notebook uses classification-style ensembles:

- **Average Ensemble:** averages the predicted probabilities from the top 3 validation models.
- **Bayesian-style Validation-Weighted Ensemble:** gives higher weight to models with stronger validation ROC-AUC. This is not claimed as full formal Bayesian Model Averaging. It is a simple Bayesian-style weighted probability ensemble that makes the weighting logic transparent for a class project.



In [6]:
# ================================
# 6. Ensemble models
# ================================

if len(top3) < 3:
    raise ValueError(
        'At least three probability-based models are needed for the top-3 ensemble. '
        f'Only found: {top3}'
    )


def get_probability_matrix(model_names, X_eval):
    """Create one matrix where each column is the positive-class probability from one model."""
    return np.column_stack([
        fitted[model_name].predict_proba(X_eval)[:, 1]
        for model_name in model_names
    ])


def evaluate_probability_ensemble(name, probabilities, y_eval, split_name, threshold=0.50):
    """Evaluate an ensemble using predicted probabilities and a default 0.50 threshold."""
    probabilities = np.clip(probabilities, 0, 1)
    predictions = (probabilities >= threshold).astype(int)
    return {
        'model': name,
        'split': split_name,
        'accuracy': accuracy_score(y_eval, predictions),
        'precision': precision_score(y_eval, predictions, zero_division=0),
        'recall': recall_score(y_eval, predictions, zero_division=0),
        'f1_score': f1_score(y_eval, predictions, zero_division=0),
        'roc_auc': roc_auc_score(y_eval, probabilities),
    }


ensemble_rows = []

# 6A. Average ensemble of the best 3 probability-based models.
for split_name, X_clean, y_split in [
    ('Validation', X_val_clean, y_val),
    ('Test', X_test_clean, y_test),
]:
    avg_probabilities = get_probability_matrix(top3, X_clean).mean(axis=1)
    ensemble_rows.append(
        evaluate_probability_ensemble('Average Ensemble (Top 3)', avg_probabilities, y_split, split_name)
    )

# 6B. BayesianRidge probability ensemble.
# The top 3 model probabilities become inputs to a BayesianRidge meta-model.
# BayesianRidge learns regularized weights and uncertainty-aware coefficients from validation predictions.
# The model is trained on validation predictions, then compared on both validation and test for the final report.
val_probability_matrix = get_probability_matrix(top3, X_val_clean)
test_probability_matrix = get_probability_matrix(top3, X_test_clean)

bayesian_ensemble_model = BayesianRidge()
bayesian_ensemble_model.fit(val_probability_matrix, y_val)

bayesian_validation_probabilities = np.clip(bayesian_ensemble_model.predict(val_probability_matrix), 0, 1)
bayesian_test_probabilities = np.clip(bayesian_ensemble_model.predict(test_probability_matrix), 0, 1)

print('BayesianRidge ensemble coefficients by model:')
for model_name, coefficient in zip(top3, bayesian_ensemble_model.coef_):
    print(f'{model_name}: {coefficient:.4f}')
print('BayesianRidge ensemble intercept:', round(float(bayesian_ensemble_model.intercept_), 4))

ensemble_rows.append(
    evaluate_probability_ensemble(
        'BayesianRidge Probability Ensemble (Top 3)',
        bayesian_validation_probabilities,
        y_val,
        'Validation',
    )
)
ensemble_rows.append(
    evaluate_probability_ensemble(
        'BayesianRidge Probability Ensemble (Top 3)',
        bayesian_test_probabilities,
        y_test,
        'Test',
    )
)

ensemble_df = pd.DataFrame(ensemble_rows)
ensemble_df[metric_columns] = ensemble_df[metric_columns].round(4)
ensemble_df.to_csv(OUTPUT / 'ensemble_metrics.csv', index=False)

combined_df = pd.concat([result_df, ensemble_df], ignore_index=True)
combined_df.sort_values(
    ['split', 'roc_auc', 'f1_score'],
    ascending=[True, False, False],
).to_csv(OUTPUT / 'combined_model_metrics.csv', index=False)

# Save the Bayesian ensemble coefficients so the report can explain how the Bayesian model blended predictions.
bayesian_weights_df = pd.DataFrame({
    'model': top3,
    'bayesian_ridge_coefficient': bayesian_ensemble_model.coef_,
})
bayesian_weights_df.to_csv(OUTPUT / 'bayesian_ensemble_coefficients.csv', index=False)

display(ensemble_df)
display(combined_df.sort_values(['split', 'roc_auc', 'f1_score'], ascending=[True, False, False]))


BayesianRidge ensemble coefficients by model:
Gradient Boosting: 0.7443
Random Forest: 0.2135
Decision Tree: 0.1292
BayesianRidge ensemble intercept: -0.0862


,model,split,accuracy,precision,recall,f1_score,roc_auc
0,Average Ensemble (Top 3),Validation,0.7853,0.5644,0.6109,0.5867,0.8024
1,Average Ensemble (Top 3),Test,0.7797,0.5521,0.6145,0.5816,0.7901
2,BayesianRidge Probability Ensemble (Top 3),Validation,0.7916,0.6050,0.4742,0.5317,0.8066
3,BayesianRidge Probability Ensemble (Top 3),Test,0.7745,0.5582,0.4553,0.5015,0.7922


,model,split,accuracy,precision,recall,f1_score,roc_auc
15,BayesianRidge Probability Ensemble (Top 3),Test,0.7745,0.5582,0.4553,0.5015,0.7922
5,Random Forest,Test,0.7679,0.5259,0.6955,0.5989,0.7908
7,Gradient Boosting,Test,0.7717,0.5551,0.4218,0.4794,0.7902
13,Average Ensemble (Top 3),Test,0.7797,0.5521,0.6145,0.5816,0.7901
3,Decision Tree,Test,0.7001,0.4432,0.7961,0.5694,0.7776
11,Support Vector Classifier,Test,0.7265,0.4649,0.6466,0.5409,0.7460
1,Logistic Regression,Test,0.7070,0.4410,0.6578,0.5280,0.7358
9,K-Nearest Neighbors,Test,0.7658,0.5413,0.3939,0.4559,0.7307
14,BayesianRidge Probability Ensemble (Top 3),Validation,0.7916,0.6050,0.4742,0.5317,0.8066
6,Gradient Boosting,Validation,0.7874,0.6008,0.4407,0.5084,0.8064


## Select best model, save outputs, and summarize

The best model is selected using validation ROC-AUC first and validation F1-score second. The test set is then used as the final performance check. This follows good machine learning practice because the test set should not be used to choose the model.

In [7]:
# ================================
# 7. Final model selection and saved outputs
# ================================

best_val = combined_df[combined_df['split'] == 'Validation'].sort_values(
    ['roc_auc', 'f1_score'],
    ascending=False,
).iloc[0]

best_name = best_val['model']
print('Best model selected from validation metrics:', best_name)

# Generate final test predictions for the selected model or ensemble.
if best_name == 'Average Ensemble (Top 3)':
    best_probabilities = get_probability_matrix(top3, X_test_clean).mean(axis=1)
    best_predictions = (best_probabilities >= 0.50).astype(int)
elif best_name == 'BayesianRidge Probability Ensemble (Top 3)':
    best_probabilities = bayesian_test_probabilities
    best_predictions = (best_probabilities >= 0.50).astype(int)
else:
    selected_model = fitted[best_name]
    best_predictions = selected_model.predict(X_test_clean)
    best_probabilities = get_model_score_values(selected_model, X_test_clean)

# Save confusion matrix and classification report.
cm = confusion_matrix(y_test, best_predictions)
confusion_df = pd.DataFrame(cm, index=['Actual 0', 'Actual 1'], columns=['Pred 0', 'Pred 1'])
confusion_df.to_csv(OUTPUT / 'best_model_confusion_matrix.csv')

with open(OUTPUT / 'classification_report.txt', 'w') as file:
    file.write(f'Best model selected from validation results: {best_name}\n\n')
    file.write(classification_report(y_test, best_predictions))

# Save feature importance from Gradient Boosting because it is usually one of the strongest interpretable models here.
try:
    importances = fitted['Gradient Boosting'].feature_importances_
    feature_importance = pd.DataFrame({
        'feature': feature_names,
        'importance': importances,
    }).sort_values('importance', ascending=False).head(15)
    feature_importance.to_csv(OUTPUT / 'gradient_boosting_top_feature_importance.csv', index=False)
except Exception as error:
    print('Feature importance could not be saved:', error)
    feature_importance = pd.DataFrame()

# Save charts for the report/presentation.
validation_plot = combined_df[combined_df['split'] == 'Validation'].sort_values('roc_auc', ascending=False)
plt.figure(figsize=(10, 5))
plt.barh(validation_plot['model'], validation_plot['roc_auc'])
plt.xlabel('Validation ROC-AUC')
plt.title('Validation ROC-AUC by Model')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(OUTPUT / 'validation_roc_auc_by_model.png', dpi=200, bbox_inches='tight')
plt.close()

test_plot = combined_df[combined_df['split'] == 'Test'].sort_values('f1_score', ascending=False)
plt.figure(figsize=(10, 5))
plt.barh(test_plot['model'], test_plot['f1_score'])
plt.xlabel('Test F1 Score')
plt.title('Test F1 Score by Model')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(OUTPUT / 'test_f1_by_model.png', dpi=200, bbox_inches='tight')
plt.close()

plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.title(f'Confusion Matrix: {best_name}')
plt.xticks([0, 1], ['Pred 0', 'Pred 1'])
plt.yticks([0, 1], ['Actual 0', 'Actual 1'])
for row_index in range(2):
    for col_index in range(2):
        plt.text(col_index, row_index, cm[row_index, col_index], ha='center', va='center')
plt.colorbar()
plt.tight_layout()
plt.savefig(OUTPUT / 'best_model_confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.close()

if not feature_importance.empty:
    plot_feature_importance = feature_importance.sort_values('importance')
    plt.figure(figsize=(8, 5))
    plt.barh(plot_feature_importance['feature'], plot_feature_importance['importance'])
    plt.xlabel('Importance')
    plt.title('Gradient Boosting Top Feature Importance')
    plt.tight_layout()
    plt.savefig(OUTPUT / 'feature_importance.png', dpi=200, bbox_inches='tight')
    plt.close()

# Save a compact JSON summary so the report can reference the exact split, metrics, and selected model.
summary = {
    'dataset_shape': list(df.shape),
    'split_counts': {
        'train': len(X_train),
        'validation': len(X_val),
        'test': len(X_test),
    },
    'numeric_features': numeric_features,
    'categorical_features': categorical_features,
    'top3_models_by_validation_roc_auc': top3,
    'bayesian_ridge_ensemble_coefficients': {
        model_name: float(coefficient)
        for model_name, coefficient in zip(top3, bayesian_ensemble_model.coef_)
    },
    'bayesian_ridge_ensemble_intercept': float(bayesian_ensemble_model.intercept_),
    'best_model_by_validation': str(best_name),
    'combined_metrics': combined_df.to_dict(orient='records'),
}

with open(OUTPUT / 'project_summary.json', 'w') as file:
    json.dump(summary, file, indent=2)

# Create report-ready PDF files required by the final exam deliverables.
# These are simple PDFs generated directly from the notebook outputs, so they can be uploaded to GitHub.
from matplotlib.backends.backend_pdf import PdfPages

comparison_pdf_path = REPORT / 'model_comparison_table.pdf'
report_pdf_path = REPORT / 'final_project_model_report.pdf'

comparison_table = combined_df.sort_values(['split', 'roc_auc', 'f1_score'], ascending=[True, False, False]).copy()

with PdfPages(comparison_pdf_path) as pdf:
    fig, ax = plt.subplots(figsize=(11, 8.5))
    ax.axis('off')
    ax.set_title('Validation and Test Metrics for All Models', fontsize=16, pad=20)
    table = ax.table(
        cellText=comparison_table.round(4).values,
        colLabels=comparison_table.columns,
        loc='center',
        cellLoc='center',
    )
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1, 1.4)
    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)

best_validation_row = comparison_table[comparison_table['split'] == 'Validation'].iloc[0]
best_test_rows = comparison_table[comparison_table['split'] == 'Test'].sort_values(['roc_auc', 'f1_score'], ascending=False)
best_test_row = best_test_rows.iloc[0]

report_lines = [
    'Final ML Project Report: HR Analytics Job Change Prediction',
    '',
    'Project Type: Classification',
    'Goal: Predict whether a candidate is likely to look for a job change.',
    '',
    'Approach:',
    'The dataset was split into 70% training, 15% validation, and 15% test sets using stratification.',
    'Preprocessing was fit only on the training set to avoid data leakage.',
    'Numeric features were imputed and scaled. Nominal categorical fields were one-hot encoded.',
    'Ordered fields such as education level, experience, last_new_job, and company_size were converted into numeric meaning.',
    '',
    'Required Models Trained:',
    'Logistic Regression, Decision Tree, Random Forest, Gradient Boosting, K-Nearest Neighbors, and Support Vector Classifier.',
    '',
    'Model Selection:',
    f'The best validation model was {best_validation_row["model"]} with validation ROC-AUC = {best_validation_row["roc_auc"]} and F1 = {best_validation_row["f1_score"]}.',
    f'The strongest final test ROC-AUC result was {best_test_row["model"]} with test ROC-AUC = {best_test_row["roc_auc"]} and F1 = {best_test_row["f1_score"]}.',
    '',
    'Why ROC-AUC and F1 Matter:',
    'Accuracy alone can be misleading because the target class is imbalanced.',
    'ROC-AUC shows ranking quality across thresholds, while F1 balances precision and recall for the positive job-change class.',
    '',
    'Ensemble Discussion:',
    'The Average Ensemble combined the top 3 validation models by averaging their positive-class probabilities.',
    'The BayesianRidge Probability Ensemble used the same top 3 model probabilities as inputs and learned regularized BayesianRidge blending coefficients from the validation set.',
    'The ensemble comparison is useful because it shows whether combining models improves generalization or only adds complexity.',
    '',
    'Final Recommendation:',
    'Use the validation-selected model as the official final model, then discuss the test metrics as the unbiased final check.',
    'If an ensemble performs better on both validation and test, it can be justified. If not, the simpler individual model should be preferred.',
]

with PdfPages(report_pdf_path) as pdf:
    fig, ax = plt.subplots(figsize=(8.5, 11))
    ax.axis('off')
    ax.text(0.03, 0.97, '\n'.join(report_lines), va='top', ha='left', fontsize=10, wrap=True)
    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)

print('Saved outputs to:', OUTPUT)
print('Saved PDF comparison table to:', comparison_pdf_path)
print('Saved PDF report draft to:', report_pdf_path)
display(confusion_df)
display(combined_df)
if not feature_importance.empty:
    display(feature_importance)


Best model selected from validation metrics: BayesianRidge Probability Ensemble (Top 3)
Saved outputs to: /content/output
Saved PDF comparison table to: /content/report/model_comparison_table.pdf
Saved PDF report draft to: /content/report/final_project_model_report.pdf


,Pred 0,Pred 1
Actual 0,1900,258
Actual 1,390,326


,model,split,accuracy,precision,recall,f1_score,roc_auc
0,Logistic Regression,Validation,0.7098,0.4439,0.6457,0.5261,0.7356
1,Logistic Regression,Test,0.7070,0.4410,0.6578,0.5280,0.7358
2,Decision Tree,Validation,0.7119,0.4559,0.8006,0.5810,0.7869
3,Decision Tree,Test,0.7001,0.4432,0.7961,0.5694,0.7776
4,Random Forest,Validation,0.7665,0.5252,0.6695,0.5886,0.8011
5,Random Forest,Test,0.7679,0.5259,0.6955,0.5989,0.7908
6,Gradient Boosting,Validation,0.7874,0.6008,0.4407,0.5084,0.8064
7,Gradient Boosting,Test,0.7717,0.5551,0.4218,0.4794,0.7902
8,K-Nearest Neighbors,Validation,0.7665,0.5466,0.3766,0.4459,0.7412
9,K-Nearest Neighbors,Test,0.7658,0.5413,0.3939,0.4559,0.7307


,feature,importance
0,city_development_index,0.668061
4,company_size,0.118611
2,education_level,0.050397
1,relevent_experience,0.046906
3,experience,0.030278
10,enrolled_university_Full time course,0.029391
6,training_hours_log1p,0.018034
24,company_type_Pvt Ltd,0.011325
5,last_new_job,0.009374
12,enrolled_university_no_enrollment,0.006328


## Final interpretation

Gradient Boosting and Random Forest are expected to perform strongly because this dataset contains a mix of numeric, ordinal, and categorical patterns. Tree-based models are good at capturing non-linear relationships, such as how experience, company size, education level, and training hours may interact with the chance of job change.

For this project, the best model is selected using validation ROC-AUC first because ROC-AUC measures how well the model ranks positive and negative cases across thresholds. F1-score is also important because the problem is imbalanced and a model can look good on accuracy while still missing many job-change candidates.

The ensemble models are useful comparison models, but they do not automatically guarantee the best result. If the top individual models make similar mistakes, averaging their probabilities may not improve performance much. This is an important real-world lesson: model complexity should be justified by better validation and test results, not by the assumption that an ensemble is always superior.

For the final report, use `output/combined_model_metrics.csv` as the main comparison table and explain the selected model based on validation ROC-AUC, validation F1-score, and final test performance.
